# Function tools and structured output with OpenAI GPT on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Client-side function tools and structured output on the Responses API, on
`bedrock-runtime`, with GPT-6 Sol (`us.openai.gpt-6-sol`). With a function tool you run
the tool yourself: the model asks for a call, your code runs it and sends the result
back, and the loop repeats until the model answers. The endpoint, the model table and
Chat Completions are in [`01-responses-api-core`](01-responses-api-core.ipynb).

**Covered:** a tool loop, several calls in one turn, forcing a tool, structured output,
and structured output from an image.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK, and uses boto3 to list the
inference profiles, because `bedrock-runtime` has no `GET /models`. From
`_shared/bedrock.py`, this repository's own helper module, it takes a lenient JSON
parser and the sample invoice.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import INVOICE_FIELDS, invoice_data_url, parse_json_lenient

REGION = "us-east-1"
MODEL = "us.openai.gpt-6-sol"

# bedrock-runtime's OpenAI-compatible path, with a short-term Bedrock API key minted
# from your AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime has no GET /models, so check the inference profiles instead.
profiles = {
    p["inferenceProfileId"]
    for page in boto3.client("bedrock", region_name=REGION).get_paginator("list_inference_profiles").paginate()
    for p in page["inferenceProfileSummaries"]
}
print(f"{MODEL} {'available' if MODEL in profiles else 'no such profile in ' + REGION}")

us.openai.gpt-6-sol available


## 1. A tool loop

A Responses function tool is flat: `name`, `description` and `parameters` sit at the top
level, where Chat Completions nests them under `function`. The model asks for a call
with a `function_call` item. Send that item back, followed by a `function_call_output`
with the same `call_id`, and ask again. Check `status` before reading the output items,
because a turn cut off at `max_output_tokens` still carries its `function_call`, whose
arguments stop mid-JSON; the loop raises there rather than guess the rest of a value.
Then check each call's arguments before running it, because a call only says what the
model wants. The loop stops after five rounds, so a model that keeps calling tools
cannot run forever.

In [2]:
STOCK = {"A-100": 42, "B-200": 0}


def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    if sku.upper() not in STOCK:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": STOCK[sku.upper()]}


TOOLS = [{
    "type": "function",
    "name": "lookup_inventory",
    "description": "Current stock level for a SKU.",
    "parameters": {
        "type": "object",
        "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
        "required": ["sku"],
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t for t in TOOLS if t["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [3]:
def ask(question: str) -> str:
    """Run the tool loop for one question and return the model's answer."""
    items = [{"role": "user", "content": question}]
    for round_no in range(1, 6):  # bounded, so a model that keeps calling tools stops
        response = client.responses.create(
            model=MODEL, input=items, tools=TOOLS, store=False, max_output_tokens=2000
        )
        # A turn cut off at max_output_tokens still carries its function_call, with
        # arguments that stop mid-JSON, and output_text that is empty or partial. Stop
        # here, because every branch below would otherwise act on a fragment.
        if response.status != "completed":
            raise RuntimeError(f"turn {round_no} {response.status}: {response.incomplete_details}")
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text
        items += [call.model_dump(exclude_none=True) for call in calls]
        for call in calls:
            # Strict parse: the lenient one closes an unterminated string and brace,
            # which would hand run_tool a shorter value than the model sent.
            try:
                args = json.loads(call.arguments or "{}")
            except json.JSONDecodeError as exc:
                args, result = call.arguments, {"error": f"arguments are not valid JSON: {exc}"}
            else:
                result = run_tool(call.name, args)
            print(f"round {round_no}: {call.name}({args}) -> {result}")
            items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
    raise RuntimeError("no answer after 5 rounds of tool calls")


print("answer:", ask("How many units of SKU A-100 do we have?").strip())

round 1: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: We have 42 units of SKU A-100.


## 2. Several calls in one turn

One turn can carry several `function_call` items, and the loop answers each one by its
`call_id`. `parallel_tool_calls=False` limits a turn to one call. `max_tool_calls` is
accepted and, in September 2026 runs, not enforced: a limit of 1 still returned five
calls. Count the calls you run in your own loop.

In [4]:
print("answer:", ask("How many units of SKU A-100 and SKU B-200 do we have?").strip())

round 1: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}
round 1: lookup_inventory({'sku': 'B-200'}) -> {'sku': 'B-200', 'quantity': 0}


answer: We have **42 units** of SKU A-100 and **0 units** of SKU B-200.


## 3. Forcing a tool

`tool_choice` takes `auto` (the default), `none`, `required` or a named function, and
all four worked on GPT-6 Sol in September 2026. Forcing a named function makes the model
fill in that function's arguments, which is one way to get structured fields out of
text. Check that the call came back before you use it.

In [5]:
response = client.responses.create(
    model=MODEL,
    input="Is SKU B-200 in stock?",
    tools=TOOLS,
    tool_choice={"type": "function", "name": "lookup_inventory"},
    store=False,
    max_output_tokens=1000,
)
calls = [item for item in response.output if item.type == "function_call"]
if not calls:
    raise RuntimeError(f"the forced call did not come back (status={response.status})")
print(f"{calls[0].name}({calls[0].arguments})")

lookup_inventory({"sku":"B-200"})


## 4. Structured output

`text={"format": {"type": "json_schema", ...}}` with `strict: True` makes the answer
JSON in that shape. List every property in `required` and set `additionalProperties` to
`false`; a strict function tool without them returns 400 `'additionalProperties' is
required to be supplied and to be false`. Check `status` first, because a cut-off answer
is cut-off JSON. A schema controls the shape and has no say over the values, so the cell
checks each value against the source text.

In [6]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
FORMAT = {"format": {"type": "json_schema", "name": "invoice", "schema": SCHEMA, "strict": True}}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
INSTRUCTION = ("Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the total "
               "as a number without a thousands separator, so 2,345.60 becomes 2345.6.")

response = client.responses.create(model=MODEL, input=f"{INSTRUCTION}\n\n{SOURCE}", text=FORMAT, max_output_tokens=2000, store=False)
if response.status != "completed":
    raise RuntimeError(f"incomplete output (status={response.status}); raise max_output_tokens")

invoice = parse_json_lenient(response.output_text)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in INVOICE_FIELDS.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 5. Structured output from an image

The same `text` format works when the input is an image. The invoice image ships with
this repository and holds the same four fields as the text above, so every value can be
checked.

In [7]:
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": [
        {"type": "input_image", "image_url": invoice_data_url()},
        {"type": "input_text", "text": INSTRUCTION},
    ]}],
    text=FORMAT,
    max_output_tokens=2000,
    store=False,
)
if response.status != "completed":
    raise RuntimeError(f"incomplete output (status={response.status}); raise max_output_tokens")

invoice = parse_json_lenient(response.output_text)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in INVOICE_FIELDS.items() if invoice.get(k) != v}
print("values match the image" if not wrong else f"values that do NOT match the image: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the image


## Things that differ

What trips people up with these models, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Tool shape | The Chat Completions shape, nested under `function`, returns 400 ``invalid request body: Invalid 'tools': missing field `name` `` | Write the flat shape for the Responses API (section 1). |
| Returning a result | 400 `No tool call found for function call output with call_id …` when the `function_call` item is not in the input | Send the `function_call` item back, then its output with the same `call_id`. |
| `max_tool_calls` | Accepted and not enforced: a limit of 1 still returned five calls (September 2026) | Count calls in your own loop, or set `parallel_tool_calls=False`. |
| Strict function tools | 400 `Invalid schema for function 'lookup_inventory': In context=(), 'additionalProperties' is required to be supplied and to be false.` | Set `additionalProperties: false` and list every property in `required`. |
| Function tools on Chat Completions | 400 `Function tools with reasoning_effort are not supported for us.openai.gpt-6-sol in /v1/chat/completions.` | Use the Responses API, or send `reasoning_effort="none"`. GPT-6.1 Sol refuses `none`, so use the Responses API. |
| Arguments | A call can name a SKU that does not exist or leave a field out | Validate each call before running it, as `run_tool` does. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Truncated output | `status` is `incomplete` with `incomplete_details.reason` `max_output_tokens`. A structured answer is cut-off JSON, and a `function_call` still arrives: at a 16-token cap in October 2026 its arguments were `{"sku":"A` and a lenient parse read them as `{"sku": "A"}` | Check `status` before you read the output items, and raise `max_output_tokens`. |
| The open-weight gpt-oss models | Tool choice and structured output behave differently | See [`14-openai-gpt-oss/01`](../14-openai-gpt-oss/01-gpt-oss-on-both-endpoints.ipynb). |

## Next

- The Responses API and the model list:
  [`01-responses-api-core`](01-responses-api-core.ipynb)
- A tool that Bedrock runs for you, web search:
  [`02-web-search-and-grounding`](02-web-search-and-grounding.ipynb)
- Caching the instructions and tool definitions of an agent:
  [`04-prompt-caching-and-cost`](04-prompt-caching-and-cost.ipynb)
- The open-weight gpt-oss models:
  [`14-openai-gpt-oss/01`](../14-openai-gpt-oss/01-gpt-oss-on-both-endpoints.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)